In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import pymorphy3
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import joblib

In [2]:
df = pd.read_csv('../data/processed/cleaned_vacancies.csv')
df.head(2)

,vacancy_id,title,author_name,description,city,salary_min,salary_max,requirements,conditions,metro,currency,experience_min,experience_max,tags,remote_type,time_type,author_id
0,49313809,Golang Developer (Кипр),Space307,Мы в Space307 разрабатываем международную торг...,Санкт-Петербург,251322.0,NaN,"Программист, разработчик",Условия обсуждаются на собеседовании,NaN,RUB,3,6.0,"docker, golang, redis, английский язык, kafka",OFFICE,FULL,c266cc48-8be0-4b5a-8e4f-03b62a9a456c
1,48813842,Е-mail маркетолог,Монополия,С 2015 года наш IT блок меняет рынок автотранс...,Санкт-Петербург,60900.0,NaN,Менеджер по маркетингу и рекламе,Условия обсуждаются на собеседовании,NaN,RUB,1,3.0,"грамотность, написание текстов, грамотная речь...",OFFICE,FULL,3c6612a4-9edd-4df6-b415-65a8d9ec5af9


In [3]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    return text

def create_super_string(row):
    # 1. Очищаем нужные поля
    title = clean_text(row.get('title', ''))
    company = clean_text(row.get('author_name', ''))
    desc = clean_text(row.get('description', ''))
    reqs = clean_text(row.get('requirements', ''))
    tags = clean_text(row.get('tags', ''))

    # 2. Исключаем заглушки
    cond = clean_text(row.get('conditions', ''))
    if "Условия обсуждаются на собеседовании" in cond:
        cond = ""
    
    
    super_string_parts = [
        title, title, title,
        company,
        tags, tags,
        desc,
        reqs,
        cond,
    ]
    
    return " ".join(filter(None, super_string_parts))

df['super_string'] = df.apply(create_super_string, axis=1)

In [4]:
nltk.download('stopwords', quiet=True)

russian_stopwords = set(stopwords.words('russian'))

custom_stops = {'вакансия', 'компания', 'искать', 'опыт', 'работа', 'год'}
russian_stopwords.update(custom_stops)

morph = pymorphy3.MorphAnalyzer()

In [5]:
def lemmatize_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Оставляем только кириллицу и латиницу
    words = re.findall(r'[а-яёa-z]+', text)
    
    lemmatized_words = []
    for word in words:
        if word not in russian_stopwords:

            normal_form = morph.parse(word)[0].normal_form
            
            if normal_form not in russian_stopwords:
                lemmatized_words.append(normal_form)
                
    return " ".join(lemmatized_words)

In [6]:
tqdm.pandas(desc="Лемматизация суперстроки")

df['super_string_lemmatized'] = df['super_string'].progress_apply(lemmatize_text)

Лемматизация суперстроки: 100%|██████████| 47325/47325 [09:16<00:00, 85.04it/s] 


In [7]:
df.to_csv('../data/processed/lemmatized_vacancies.csv', index=False)
print(f"Сохранено {len(df)} вакансий.")
df.head(3)

Сохранено 47325 вакансий.


,vacancy_id,title,author_name,description,city,salary_min,salary_max,requirements,conditions,metro,currency,experience_min,experience_max,tags,remote_type,time_type,author_id,super_string,super_string_lemmatized
0,49313809,Golang Developer (Кипр),Space307,Мы в Space307 разрабатываем международную торг...,Санкт-Петербург,251322.0,NaN,"Программист, разработчик",Условия обсуждаются на собеседовании,NaN,RUB,3,6.0,"docker, golang, redis, английский язык, kafka",OFFICE,FULL,c266cc48-8be0-4b5a-8e4f-03b62a9a456c,golang developer (кипр) golang developer (кипр...,golang developer кипр golang developer кипр go...
1,48813842,Е-mail маркетолог,Монополия,С 2015 года наш IT блок меняет рынок автотранс...,Санкт-Петербург,60900.0,NaN,Менеджер по маркетингу и рекламе,Условия обсуждаются на собеседовании,NaN,RUB,1,3.0,"грамотность, написание текстов, грамотная речь...",OFFICE,FULL,3c6612a4-9edd-4df6-b415-65a8d9ec5af9,е-mail маркетолог е-mail маркетолог е-mail мар...,mail маркетолог mail маркетолог mail маркетоло...
2,49413720,Оператор call-центра (удаленно),Eden Springs,Что нужно будет делать: Принимать входящие зв...,Санкт-Петербург,NaN,NaN,"Оператор call-центра, специалист контактного ц...",Условия обсуждаются на собеседовании,NaN,RUB,1,3.0,"клиентоориентированность, ориентация на резуль...",REMOTE,FULL,97d3e7aa-10fc-4825-9f38-ef5bb0279eaa,оператор call-центра (удаленно) оператор call-...,оператор call центр удалённый оператор call це...


In [8]:
vectorizer = TfidfVectorizer(max_features=10000)

print("Начинаем векторизацию...")
tfidf_matrix = vectorizer.fit_transform(df['super_string_lemmatized'])
print(f"Размер полученной матрицы: {tfidf_matrix.shape}")

# 3. Сохраняем матрицу векторов
sp.save_npz('../data/processed/tfidf_matrix.npz', tfidf_matrix)

# 4. Сохраняем векторизатор
joblib.dump(vectorizer, '../data/processed/tfidf_vectorizer.pkl')

columns_to_keep = ['vacancy_id', 'title', 'author_name', 'salary_min', 'salary_max', 'city', 'experience_min', 'remote_type']
df_meta = df[columns_to_keep]

df_meta.to_csv('../data/processed/vacancies_meta.csv', index=False)
print("Матрица и метаданные успешно сохранены!")

Начинаем векторизацию...
Размер полученной матрицы: (47325, 10000)
Матрица и метаданные успешно сохранены!
